In [1]:
import pandas as pd
import numpy as np

In [2]:
panel = pd.read_parquet("../data/pipeline/panel.parquet")

In [ ]:
panel_no_negrisk = pd.read_parquet("../data/pipeline/panel_no_negrisk.parquet")

token_id  \
market_id t                                                               
1004538   1766516412  1013473364359957167552629422664460003941737285...   
          1766520011  1013473364359957167552629422664460003941737285...   
          1766523613  1013473364359957167552629422664460003941737285...   
          1766527212  1013473364359957167552629422664460003941737285...   
          1766530812  1013473364359957167552629422664460003941737285...   

                          p price_bins  yes_outcomes  \
market_id t                                            
1004538   1766516412  0.410  0.40-0.45         False   
          1766520011  0.420  0.40-0.45         False   
          1766523613  0.485  0.45-0.50         False   
          1766527212  0.460  0.45-0.50         False   
          1766530812  0.440  0.40-0.45         False   

                                            createdAt  \
market_id t                                             
1004538   1766516412 2025-12-23 16:19:33.493952+00:00   
          1766520011 2025-12-23 16:19:33.493952+00:00   
          1766523613 2025-12-23 16:19:33.493952+00:00   
          1766527212 2025-12-23 16:19:33.493952+00:00   
          1766530812 2025-12-23 16:19:33.493952+00:00   

                                    closedTime  time_to_resolution    ttr_bin  \
market_id t                                                                     
1004538   1766516412 2026-01-01 18:48:41+00:00          215.808060  1-2 weeks   
          1766520011 2026-01-01 18:48:41+00:00          214.808334  1-2 weeks   
          1766523613 2026-01-01 18:48:41+00:00          213.807785  1-2 weeks   
          1766527212 2026-01-01 18:48:41+00:00          212.808060  1-2 weeks   
          1766530812 2026-01-01 18:48:41+00:00          211.808060  1-2 weeks   

                      negRisk  
market_id t                    
1004538   1766516412    False  
          1766520011    False  
          1766523613    False  
          1766527212    False  
          1766530812    False

In [17]:
panel_negrisk = pd.read_parquet("../data/pipeline/panel_negrisk.parquet")

In [3]:
# an alternative to panda's insert(args) since it throws an error if the column already exists
# same logic but instead of df.insert, we use insert_or_replace(df, args)
def insert_or_replace(df, position, name, values):
    if name in df.columns:
        df.drop(columns=name, inplace=True)
    df.insert(position, name, values)


# just to ensure our df isnt getting bloated anywhere
def check_memory(df):
    print(df.memory_usage(deep=True) / 1e6) # MB per column
    print()
    print(f"Total memory used: {df.memory_usage(deep=True).sum()/1e6}")


# gives the position to insert columns after a specified column
from typing import cast

# for some reason pylance never lets me use .get_loc() + 1
def nextpos(df, col:str)->int:
    pos = cast(int, df.columns.get_loc(col))
    return pos + 1

# allows us to deal with the edge cases of p = 0 and p = 1 (where 1-p=0)
# avoding log2(0) undefined problems
def binary_entropy(p):
    p = np.asarray(p, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        term1 = np.where(p > 0, p * np.log2(p), 0.0)
        term2 = np.where(p < 1, (1 - p) * np.log2(1 - p), 0.0)
    return -(term1 + term2)


In [22]:
# another cell combining pipeline
def entropy_calculations(input):
    market_entropy = binary_entropy(input["p"])

    insert_or_replace(input, nextpos(input, "p"), "market_entropy", market_entropy) 
    # market entropy here isnt overall entropy (thats for the whole
    # distribution, a single number), its predictive uncertainty
    # at that point

    #### CALCULATING ENTROPY H(Y)
    ## The entropy of market outcomes before we look at prices. It is just the overall uncertainty of outcomes in our sample.

    # before i just took the mean of yes_outcomes, but the problem with that
    # is that markets that are open longer simply have more observations
    # and thus get sampled more
    # this gets us back to our old 50k market gamma data and takes the
    # first value from each, since for a market, the yes_outcomes is
    # gonna be the same anyways
    p_yes = input.groupby("market_id")["yes_outcomes"].first().mean()

    entropy_Y = -(p_yes * np.log2(p_yes) + (1 - p_yes) * np.log2(1 - p_yes))


    #### CALCULATING CONDITIONAL ENTROPY H(Y|P)
    ## Now we calculate the entropy within each price bin and take the expected value to get
    ##  the uncertainty of outcomes within each price bin.

    # get a smaller df thats sorted by bins
    binned = input.groupby("price_bins", observed=True)["yes_outcomes"].mean()
    binned = binned.to_frame() # it isnt automatically a df


    # two outcomes y/n with probability p and 1-p, like a bernoulli
    binned["entropy"] = binary_entropy(binned["yes_outcomes"])


    # get conditional entropy, i.e. expectations over the bins
    binned["weights"] = input["price_bins"].value_counts(normalize = True)
    binned["w_entropy"] = binned["weights"] * binned["entropy"]


    # weights into entropy gives expected entropy
    # summed over price bins gives entropy of outcome given price
    cond_entropy = binned["w_entropy"].sum()

    MI = entropy_Y-cond_entropy

    uncertainty_reduced = MI * 100/entropy_Y

    # round causes floating point issues for the uncertainty calc
    print(f"""Entropy: {entropy_Y.round(5)}
    Conditional entropy: {cond_entropy.round(5)}
    Mutual information: {MI.round(5)}
    Uncertainty explained by market prices: {uncertainty_reduced:.3f}%""")

    return entropy_Y, cond_entropy, MI, uncertainty_reduced


In [23]:
entropy_Y, cond_entropy, MI, uncertainty_reduced = entropy_calculations(panel)

Entropy: 0.86347
    Conditional entropy: 0.40019
    Mutual information: 0.46328
    Uncertainty explained by market prices: 53.654%


In [24]:
entropy_Y_nonegrisk, cond_entropy_nonegrisk, MI_nonegrisk, uncertainty_reduced_nonegrisk = entropy_calculations(panel_no_negrisk)

Entropy: 0.94988
    Conditional entropy: 0.45245
    Mutual information: 0.49743
    Uncertainty explained by market prices: 52.367%


In [25]:
entropy_Y_negrisk, cond_entropy_negrisk, MI_negrisk, uncertainty_reduced_negrisk = entropy_calculations(panel_negrisk)

Entropy: 0.78834
    Conditional entropy: 0.3729
    Mutual information: 0.41544
    Uncertainty explained by market prices: 52.698%
